The [previous section](/aiml-common/lectures/nlp/nlp-introduction/word2vec) set up the skip-gram objective with a full softmax over the vocabulary. In this section you train skip-gram with *negative sampling* (SGNS), the variant that made word2vec practical [@Mikolov2013-bz], on text8: the first 15 million words of a cleaned English Wikipedia dump. You then check what the vectors learned with nearest neighbors and with the analogy test of Mikolov et al. [@Mikolov2013-is].

Each word $w$ gets two vectors: an input vector $\mathbf v_w$, used when $w$ is the center word, and an output vector $\mathbf u_w$, used when $w$ is a context word. For a center word $c$ and an observed context word $o$, SGNS replaces the softmax over all $|V|$ words with $K$ binary decisions against random *noise* words $w_1, \dots, w_K$:

$$
\ell(c, o) = -\log \sigma(\mathbf u_o^\top \mathbf v_c) - \sum_{k=1}^{K} \log \sigma(-\mathbf u_{w_k}^\top \mathbf v_c), \qquad w_k \sim P_n(w)
$$

The first term pushes the score of the true pair up. The second pushes the scores of the noise pairs down. One update touches $K+2$ vectors instead of all $|V|$ output vectors, which is what lets the model train on billions of words.

In [ ]:
import warnings

warnings.filterwarnings("ignore")

import collections
import time
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import disable_progress_bars, load_dataset

disable_progress_bars()

torch.manual_seed(0)
np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

## Corpus and vocabulary

text8 is a single string of lowercase words with punctuation and numbers removed. Words that occur fewer than 5 times are dropped, as in the original word2vec code: their vectors would be trained on too few examples to mean anything, and they make up most of the vocabulary.

In [ ]:
text = load_dataset("afmck/text8", split="train")[0]["text"]
words = text.split()

MIN_COUNT = 5
counts = collections.Counter(words)
vocab = [w for w, c in counts.most_common() if c >= MIN_COUNT]
word_to_id = {w: i for i, w in enumerate(vocab)}
freq = np.array([counts[w] for w in vocab], dtype=np.float64)

ids = np.array([word_to_id[w] for w in words if w in word_to_id], dtype=np.int64)
print(f"tokens in corpus      : {len(words):,}")
print(f"vocabulary (count>=5) : {len(vocab):,} of {len(counts):,} distinct words")
print(f"tokens kept           : {len(ids):,}")
print("most frequent         :", vocab[:10])

## Subsampling and the noise distribution

Frequent words carry little information about their neighbors: almost every word appears next to *the*. Mikolov et al. discard each occurrence of word $w$ with probability $1 - \sqrt{t / f(w)}$, where $f(w)$ is the word's share of the corpus and $t$ is a threshold [@Mikolov2013-bz]. Words with $f(w) < t$ are always kept. A fresh subsample is drawn every epoch, so every occurrence gets some chance to be used.

Noise words are drawn from the unigram distribution raised to the power $3/4$, $P_n(w) \propto f(w)^{3/4}$. The exponent flattens the distribution: rare words are drawn more often than their frequency alone would give, and frequent words less often.

In [ ]:
SUBSAMPLE_T = 1e-4
f = freq / freq.sum()
keep_prob = np.minimum(1.0, np.sqrt(SUBSAMPLE_T / f))
noise = f**0.75
noise /= noise.sum()

for w in ["the", "of", "king", "algebra"]:
    i = word_to_id[w]
    print(f"{w:>8}: f = {f[i]:.2e}  keep = {keep_prob[i]:.3f}  P_n / f = {noise[i] / f[i]:.2f}")
print(f"expected share of tokens kept per epoch: {(keep_prob * f).sum():.2f}")

## Training pairs

Pairs are built from the subsampled token stream with a window of up to $m = 5$ words on each side. For each center position the window size $b$ is drawn uniformly from $1, \dots, m$, as in the original implementation. A context word at distance $k$ is therefore used with probability $(m - k + 1)/m$, so near neighbors count more than far ones.

In [ ]:
WINDOW = 5


def make_pairs(ids_gpu, keep_prob_gpu, window):
    # One epoch of (center, context) pairs: subsample, then a random window per center.
    stream = ids_gpu[torch.rand(len(ids_gpu), device=ids_gpu.device) < keep_prob_gpu[ids_gpu]]
    b = torch.randint(1, window + 1, (len(stream),), device=stream.device)
    centers, contexts = [], []
    for k in range(1, window + 1):
        right = b[:-k] >= k  # center i, context i + k
        centers.append(stream[:-k][right])
        contexts.append(stream[k:][right])
        left = b[k:] >= k  # center i + k, context i
        centers.append(stream[k:][left])
        contexts.append(stream[:-k][left])
    centers, contexts = torch.cat(centers), torch.cat(contexts)
    perm = torch.randperm(len(centers), device=stream.device)
    return centers[perm], contexts[perm]


ids_gpu = torch.from_numpy(ids).to(device)
keep_prob_gpu = torch.from_numpy(keep_prob).float().to(device)
noise_gpu = torch.from_numpy(noise).float().to(device)

c, o = make_pairs(ids_gpu, keep_prob_gpu, WINDOW)
print(f"pairs in one epoch: {len(c):,}")
print("examples:", [(vocab[a], vocab[b]) for a, b in zip(c[:5].tolist(), o[:5].tolist())])

## Model

The model is two embedding tables of size $|V| \times d$, one for input vectors and one for output vectors. The score of a pair is a dot product, and the loss is $\ell(c, o)$ averaged over a batch. Input vectors start small and random and output vectors start at zero, as in the original code. Both tables use sparse gradients: a batch touches only the rows of the words in it, and `SparseAdam` updates only those rows.

In [ ]:
class SkipGramNS(nn.Module):
    def __init__(self, vocab_size, dim):
        super().__init__()
        self.inp = nn.Embedding(vocab_size, dim, sparse=True)  # v_w
        self.out = nn.Embedding(vocab_size, dim, sparse=True)  # u_w
        nn.init.uniform_(self.inp.weight, -0.5 / dim, 0.5 / dim)
        nn.init.zeros_(self.out.weight)

    def forward(self, center, context, negatives):
        v = self.inp(center)  # (B, d)
        u_pos = self.out(context)  # (B, d)
        u_neg = self.out(negatives)  # (B, K, d)
        pos = (v * u_pos).sum(-1)
        neg = torch.bmm(u_neg, v.unsqueeze(-1)).squeeze(-1)
        return -(F.logsigmoid(pos) + F.logsigmoid(-neg).sum(-1)).mean()

## Training

Each epoch draws a new subsample and a new set of pairs, then runs through them in batches. The $K = 5$ noise words for every pair are drawn from $P_n$ at the time of the batch. The noise words are not checked against the true context word: with a vocabulary of this size a collision is rare and does little harm.

In [ ]:
DIM, NEG, BATCH, EPOCHS, LR = 200, 5, 4096, 5, 2e-3

model = SkipGramNS(len(vocab), DIM).to(device)
optimizer = torch.optim.SparseAdam(model.parameters(), lr=LR)

history = []
for epoch in range(EPOCHS):
    start = time.time()
    centers, contexts = make_pairs(ids_gpu, keep_prob_gpu, WINDOW)
    running, n = 0.0, 0
    for i in range(0, len(centers), BATCH):
        c, o = centers[i : i + BATCH], contexts[i : i + BATCH]
        negatives = torch.multinomial(noise_gpu, len(c) * NEG, replacement=True).view(len(c), NEG)
        loss = model(c, o, negatives)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        running += loss.item()
        n += 1
        if n % 2000 == 0:
            history.append(running / 2000)
            running = 0.0
    print(f"epoch {epoch + 1}: {len(centers):,} pairs, last loss {history[-1]:.3f}, {time.time() - start:.0f} s")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(np.arange(1, len(history) + 1) * 2000 * BATCH / 1e6, history)
ax.set_xlabel("training pairs seen (millions)")
ax.set_ylabel("mean loss per 2000 batches")
ax.set_title("SGNS training loss on text8")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

The loss falls quickly during the first epoch, about 45 million pairs, and slowly over the remaining four. Its floor is well above zero because many observed pairs, such as (*city*, *the*), are no more likely than the noise pairs: the model cannot and should not separate them.

## Nearest neighbors

After training, the input vectors $\mathbf v_w$ are the word embeddings. Two words are close when the cosine of the angle between their vectors is large. Words that appear in similar contexts should end up close, whether or not they ever appear together.

In [ ]:
emb = F.normalize(model.inp.weight.detach(), dim=1)


def neighbors(word, k=6):
    sims = emb @ emb[word_to_id[word]]
    best = sims.topk(k + 1).indices.tolist()[1:]
    return [(vocab[i], round(sims[i].item(), 2)) for i in best]


for w in ["france", "computer", "einstein", "guitar", "january", "bank"]:
    print(f"{w:>9}:", ", ".join(f"{n} {s}" for n, s in neighbors(w)))

The neighbors of *bank* show the limit set out in the previous section. There is one vector per word, so the financial and the river senses share it, and whichever sense dominates the corpus dominates the neighbors.

## Analogies

The analogy test asks for the word $d$ that completes "$a$ is to $b$ as $c$ is to $d$" by finding the vocabulary word whose vector is closest to $\mathbf v_b - \mathbf v_a + \mathbf v_c$, excluding $a$, $b$ and $c$ themselves. Mikolov et al. released about 19,500 such questions: 8,869 semantic ones (capitals, currencies, family relations) and 10,675 syntactic ones (plurals, tenses, comparatives) [@Mikolov2013-is]. A question counts as correct only if the top-ranked word is exactly the expected one.

In [ ]:
URL = "https://raw.githubusercontent.com/tmikolov/word2vec/master/questions-words.txt"
lines = urllib.request.urlopen(URL).read().decode().splitlines()

sections, current = collections.defaultdict(list), None
for line in lines:
    if line.startswith(":"):
        current = line[2:]
        continue
    q = line.lower().split()
    if all(w in word_to_id for w in q):
        sections[current].append([word_to_id[w] for w in q])


def analogy_accuracy(questions, batch=1024):
    q = torch.tensor(questions, device=device)
    correct = 0
    for i in range(0, len(q), batch):
        a, b, c, d = q[i : i + batch].T
        sims = (emb[b] - emb[a] + emb[c]) @ emb.T
        for col in (a, b, c):
            sims.scatter_(1, col[:, None], -float("inf"))
        correct += (sims.argmax(1) == d).sum().item()
    return correct / len(q)


semantic = [q for s, qs in sections.items() if not s.startswith("gram") for q in qs]
syntactic = [q for s, qs in sections.items() if s.startswith("gram") for q in qs]
total = sum(1 for line in lines if not line.startswith(":"))
print(f"questions with all four words in the vocabulary: {len(semantic) + len(syntactic):,} of {total:,}")
print(f"semantic  accuracy: {analogy_accuracy(semantic):.1%}  ({len(semantic):,} questions)")
print(f"syntactic accuracy: {analogy_accuracy(syntactic):.1%}  ({len(syntactic):,} questions)")
for s in ["capital-common-countries", "family", "gram2-opposite", "gram8-plural"]:
    print(f"  {s:<26} {analogy_accuracy(sections[s]):.1%}")

In [ ]:
def solve(a, b, c, k=3):
    target = emb[word_to_id[b]] - emb[word_to_id[a]] + emb[word_to_id[c]]
    sims = emb @ target
    for w in (a, b, c):
        sims[word_to_id[w]] = -float("inf")
    return [vocab[i] for i in sims.topk(k).indices.tolist()]


for a, b, c in [("man", "king", "woman"), ("france", "paris", "italy"), ("big", "bigger", "small"), ("walking", "walked", "swimming")]:
    print(f"{a} : {b} :: {c} : ?  ->  {solve(a, b, c)}")

On text8 the model answers roughly 30% of both the semantic and the syntactic questions. Of the four categories printed, capitals of common countries score highest and opposites such as *possible* : *impossible* lowest. Two of the four questions above fail: the Italy question returns other Italian cities, and the swimming question returns unrelated words. Fifteen million words are enough to place the capitals of large countries but not to learn every regular inflection.

The offset behind the capital questions can be seen directly. The figure projects the vectors of eight countries and their capitals onto their first two principal components and draws an arrow from each country to its capital. The arrows point in roughly the same direction: the model has learned a "capital of" offset without ever being told what a capital is.

In [ ]:
pairs = [("france", "paris"), ("italy", "rome"), ("germany", "berlin"), ("spain", "madrid"),
         ("russia", "moscow"), ("japan", "tokyo"), ("china", "beijing"), ("greece", "athens")]
X = emb[[word_to_id[w] for p in pairs for w in p]].cpu().numpy()
X = X - X.mean(0)
P = X @ np.linalg.svd(X, full_matrices=False)[2][:2].T

fig, ax = plt.subplots(figsize=(6.5, 5))
for i, (country, capital) in enumerate(pairs):
    (x0, y0), (x1, y1) = P[2 * i], P[2 * i + 1]
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0), arrowprops=dict(arrowstyle="->", color="tab:red", alpha=0.6))
    ax.scatter([x0, x1], [y0, y1], c=["tab:blue", "tab:orange"], s=20)
    ax.text(x0, y0, country, fontsize=9, ha="right", va="bottom")
    ax.text(x1, y1, capital, fontsize=9, ha="left", va="top")
ax.set_title("Countries (blue) and capitals (orange), first two principal components")
ax.set_xticks([])
ax.set_yticks([])
plt.tight_layout()
plt.show()

## Limits of this run

text8 is small by word2vec standards. Mikolov et al. trained on corpora of hundreds of millions to billions of words and report higher analogy accuracy as the corpus grows [@Mikolov2013-is], so the accuracies above are a floor for the method, not its ceiling. The analogy test is also a narrow probe. Levy and Goldberg showed that SGNS implicitly factorizes a shifted matrix of pointwise mutual information between words and contexts ([NeurIPS 2014](https://papers.nips.cc/paper/5477-neural-word-embedding-as-implicit-matrix-factorization)). Levy et al. later found that much of its advantage over count-based methods comes from choices such as subsampling and the window distribution, which carry over to those methods ([TACL 2015](https://aclanthology.org/Q15-1016/)). Neither nearest neighbors nor analogies tell you how well the vectors will serve a downstream task.

## PyTorch reference

| PyTorch class | Description |
|---|---|
| [`nn.Embedding`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.Embedding.html) | A lookup table of vectors; `sparse=True` makes its gradient a sparse tensor. |
| [`torch.optim.SparseAdam`](https://docs.pytorch.org/docs/2.12/generated/torch.optim.SparseAdam.html) | Adam that updates only the rows present in a sparse gradient. |
| [`F.logsigmoid`](https://docs.pytorch.org/docs/2.12/generated/torch.nn.functional.logsigmoid.html) | Numerically stable $\log \sigma(x)$. |
| [`torch.multinomial`](https://docs.pytorch.org/docs/2.12/generated/torch.multinomial.html) | Draws indices from a categorical distribution, used here for noise words. |